# ***Assignment 16: Recommendation System (Anime Dataset)***

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

**Task 1: Data Preprocessing**

In [4]:
# Load dataset
df_anime = pd.read_csv("/content/anime.csv")

print("--- Dataset Overview ---")
print(df_anime.info())

print("\n--- Summary Statistics ---")
print(df_anime.describe())

print("\n--- Missing Values Count ---")
print(df_anime.isnull().sum())

# Handle missing values:
# Fill missing genres and types with empty string, missing ratings with median
df_clean = df_anime.copy()
df_clean["genre"] = df_clean["genre"].fillna("")
df_clean["type"] = df_clean["type"].fillna("Unknown")
df_clean["rating"] = df_clean["rating"].fillna(df_clean["rating"].median())

# Clean title formatting if necessary (e.g., HTML entities)
df_clean["name"] = (
    df_clean["name"]
    .str.replace("&#039;", "'")
    .str.replace("&quot;", '"')
    .str.replace("&amp;", "&")
)

--- Dataset Overview ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
None

--- Summary Statistics ---
           anime_id        rating       members
count  12294.000000  12064.000000  1.229400e+04
mean   14058.221653      6.473902  1.807134e+04
std    11455.294701      1.026746  5.482068e+04
min        1.000000      1.670000  5.000000e+00
25%     3484.250000      5.880000  2.250000e+02
50%    10260.500000      6.570000  1.550000e+03
75%    24794.500000      7.180000  9.437000e+03
max    34527.000000     10.000000  

# **Task 2: Feature Extraction**

In [5]:
# 1. TF-IDF Vectorization on Genre
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df_clean["genre"])

print("\nTF-IDF Matrix Shape (Genres):", tfidf_matrix.shape)

# 2. Normalize Numerical Features (Rating & Members)
scaler = MinMaxScaler()
numerical_scaled = scaler.fit_transform(df_clean[["rating", "members"]])


TF-IDF Matrix Shape (Genres): (12294, 46)


**Task 3: Recommendation System Implementation**

In [6]:
def recommend_anime(
    anime_title, top_n=5, similarity_threshold=0.2, df=df_clean, matrix=tfidf_matrix
):
    """Recommends similar anime based on genre TF-IDF cosine similarity."""
    # Search for target anime by title
    matches = df[df["name"].str.lower() == anime_title.lower()]
    if matches.empty:
        matches = df[
            df["name"].str.contains(anime_title, case=False, regex=False)
        ]

    if matches.empty:
        print(f"Anime titled '{anime_title}' not found in dataset.")
        return None

    target_idx = matches.index[0]
    target_name = df.loc[target_idx, "name"]
    print(f"\n--- Recommendations for: '{target_name}' ---")

    # Compute Cosine Similarity between target anime and all anime
    target_vec = matrix[target_idx]
    sim_scores = cosine_similarity(target_vec, matrix).flatten()

    # Filter indices matching threshold criteria
    eligible_indices = np.where(sim_scores >= similarity_threshold)[0]
    eligible_indices = [i for i in eligible_indices if i != target_idx]

    # Sort candidates by similarity score
    sorted_indices = sorted(
        eligible_indices, key=lambda x: sim_scores[x], reverse=True
    )[:top_n]

    if not sorted_indices:
        print(f"No recommendations met the threshold of {similarity_threshold}.")
        return None

    recommendations = df.iloc[sorted_indices][
        ["name", "genre", "type", "episodes", "rating", "members"]
    ].copy()
    recommendations["similarity_score"] = sim_scores[sorted_indices]

    return recommendations


# Test Recommendation Function with Target Anime
print(recommend_anime("Fullmetal Alchemist: Brotherhood", top_n=5, similarity_threshold=0.3))


# Experiment with Different Similarity Thresholds
print("\n--- Experimenting with Threshold = 0.8 (Higher Specificity) ---")
print(recommend_anime("Steins;Gate", top_n=5, similarity_threshold=0.8))

print("\n--- Experimenting with Threshold = 0.1 (Broader Pool) ---")
print(recommend_anime("Steins;Gate", top_n=5, similarity_threshold=0.1))


--- Recommendations for: 'Fullmetal Alchemist: Brotherhood' ---
                                               name  \
200                             Fullmetal Alchemist   
1558  Fullmetal Alchemist: The Sacred Star of Milos   
402       Fullmetal Alchemist: Brotherhood Specials   
879             Tales of Vesperia: The First Strike   
4264                                 Tide-Line Blue   

                                                  genre     type episodes  \
200   Action, Adventure, Comedy, Drama, Fantasy, Mag...       TV       51   
1558  Action, Adventure, Comedy, Drama, Fantasy, Mag...    Movie        1   
402   Adventure, Drama, Fantasy, Magic, Military, Sh...  Special        4   
879         Action, Adventure, Fantasy, Magic, Military    Movie        1   
4264        Action, Adventure, Drama, Military, Shounen       TV       12   

      rating  members  similarity_score  
200     8.33   600384          0.973258  
1558    7.50    87944          0.973258  
402     8.11   